# Lesson 11 — I2C 地址、ACK 和设备缺失

这是一段真实的 C++ 实验：你修改代码格，再运行，就会看到下方深色虚拟设备的输入、输出和检查结果变化。Python 只负责让 VS Code Notebook 启动 C++ 编译和实验台。

## 先运行

按顺序运行初始化格和后面的代码格。绿色 ✓ 表示本格 C++ 符合当前契约；红色 ✗ 会给出预期与实际输出。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


I2C 总线共享两根线，主机必须确认目标地址和应答。读不到设备时继续把旧值当新数据会误导上层。

## 最小知识

本机协议模拟一次事务：`I2C:0x48:READ` 读取温度。成功返回 `ACK:数值\n`；错误地址 `NACK\n`；断连/故障 `TIMEOUT\n`。


## 1. 第一个实验（Guided）

先运行。不要急着读每一行；先操作实验台，观察输入与输出之间的关系。


In [ ]:
%%academy_lab 11-i2c-sensor guided guided pass
#pragma once
#include "academy/device.hpp"
#include <string>
namespace academy {class Controller{public:explicit Controller(Device&b):b_(b){}void tick(){if(b_.input.text=="I2C:0x48:READ")b_.output.outgoing="ACK:"+std::to_string(b_.input.temperature)+"\n";}private:Device&b_;};}


## 2. 修改一次

增加一个 `I2C:0x48:ID` 响应。协议字符串是实验支架，地址/ACK/错误语义才是重点。

重新运行这一个格，再在实验台确认你的预测。


In [ ]:
%%academy_lab 11-i2c-sensor modify
#pragma once
#include "academy/device.hpp"
#include <string>
namespace academy {class Controller{public:explicit Controller(Device&b):b_(b){}void tick(){if(b_.input.text=="I2C:0x48:READ")b_.output.outgoing="ACK:"+std::to_string(b_.input.temperature)+"\n";}private:Device&b_;};}


## 3. 故障实验（Debug）

下面代码不验证地址，任何请求都能读到温度，隐藏了接错设备的错误。运行后失败是预期现象。用面板的时间线、波形、事务或终端找出失败发生在何时。


In [ ]:
%%academy_lab 11-i2c-sensor broken exercise fail
#pragma once
#include "academy/device.hpp"
#include <string>
namespace academy {class Controller{public:explicit Controller(Device&b):b_(b){}void tick(){if(b_.input.text=="I2C:0x48:READ")b_.output.outgoing="ACK:"+std::to_string(b_.input.temperature)+"\n";}private:Device&b_;};}


## 4. 你来完成（Build）

验证地址和连接状态，产生明确 ACK/NACK/TIMEOUT，并且错误事务不能伪造 reading。

默认这格会失败，失败信息是需求的一部分。先自己尝试；需要时再看 [分层 Hint](../lessons/11-i2c-sensor/hints.md)。


In [ ]:
%%academy_lab 11-i2c-sensor exercise exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {class Controller{public:explicit Controller(Device&b):b_(b){}void tick(){// TODO: 地址、ACK/NACK、TIMEOUT
}private:Device&b_;};}


## 5. 开放挑战

为多个地址设计 Driver 注册表。何时应该重试，何时应该把总线标记为不可用？

## 你刚刚真正学到了什么？

I2C 的“读取成功”包括地址匹配、应答和数据解释。Driver 必须把这些失败传给上层。

## 硬件迁移

STEP 2 需要上拉电阻、正确电压、地址扫描与逻辑分析仪验证；本机面板只模拟协议结果。

完成自己的尝试后才查看 [独立 Solution Notebook](solutions/11-i2c-sensor-solution.ipynb)。
